# Разработка LangChain приложений с локальной LLM

## Настройка окружения

In [ ]:
# Убедитесь, что выбрана среда выполнения GPU!

print("Updating package lists and installing build tools...")
# Убедитесь, что все необходимые инструменты сборки присутствуют!
!apt-get update > /dev/null
!apt-get install -y build-essential cmake > /dev/null # lib для компиляции кода на cpp
print("Build tools check/install complete.")

# --- Проверка версии системы ---
print("\nChecking system versions...")
!nvcc --version # nvidia
!gcc --version # cpp
!python --version
print("System versions checked.")

print("\nInstalling SPECIFIC version of llama-cpp-python (0.2.73) with GPU support...")
# Закрепление версии часто помогает избежать проблем сборки с абсолютно последним релизом
# Убедитесь, что флаги CMAKE передаются правильно
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.73 --no-cache-dir --quiet
print("llama-cpp-python installation attempt complete.")

print("\nInstalling other dependencies (Langchain, etc.)...")
# Установите остальное
!pip install langchain langchain_community huggingface_hub duckduckgo-search tiktoken --quiet
print("Other dependencies installed.")

# --- Проверьте импорт llama-cpp-python ---
print("\nVerifying llama-cpp-python import...")
try:
    import llama_cpp
    print("✅ Successfully imported llama_cpp.")
    # Необязательно: Распечатать местоположение, чтобы увидеть, где оно оказалось
    # print(f" llama_cpp path: {llama_cpp.__file__}")
except ImportError as e:
    print(f"❌ Failed to import llama_cpp after installation: {e}")
    print(" This suggests the installation failed. Check the detailed pip install output above for errors.")
    print(" Try 'Runtime -> Factory reset runtime' and run this cell again.")
    print(" If it still fails, the Colab environment might be incompatible with this version/build method right now.")
    raise ImportError("llama-cpp-python failed to install correctly.") from e

Updating package lists and installing build tools...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Build tools check/install complete.

Checking system versions...
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Python 3.12.11
System versions checked.

Installing SPECIFIC version of llama-cpp-python (0.2.73) with GPU support...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 MB 13.7 MB/s eta 0:00:00
  Installing build dependencies ... done

## Загрузка локальной модели

In [ ]:
# --- Загрузить модель ---
from huggingface_hub import hf_hub_download
import os

model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF" # GGUF — специальный формат для быстрой загрузки и соххранения LLM
model_basename = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"  # 4-bit quantization

model_path = os.path.join(os.getcwd(), model_basename)
if not os.path.exists(model_path):
    print(f"\nDownloading model '{model_basename}' from '{model_name_or_path}'...")
    try:
        model_path = hf_hub_download( # скачиваем модель
            repo_id=model_name_or_path,
            filename=model_basename,
            local_dir=os.getcwd(),
            local_dir_use_symlinks=False
        )
        print("Model download complete.")
    except Exception as e:
        print(f"Error downloading model: {e}")
        raise e
else:
    print(f"\nModel '{model_basename}' already exists locally.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


mistral-7b-instruct-v0.2.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

Model download complete.


## Инициализация модели

In [ ]:
# --- Создание локальной LLM (используя ChatLlamaCpp) ---
from langchain_community.chat_models import ChatLlamaCpp

llm = None  # Инициализируем llm как None
try:
    llm = ChatLlamaCpp(
        model_path=model_path,
        temperature=0.1,
        max_tokens=1024,
        n_ctx=2048,
        n_gpu_layers=35,  # Отрегулируйте на основе видеопамяти вашего графического процессора (T4 обычно справляется с ~35)
        n_batch=512,
        verbose=False,
        f16_kv=True,  # Необязательно: используйте, если памяти мало
    )
    print("\n✅ ChatLlamaCpp Model Instantiated Successfully.")

    # Выполнить быстрый тестовый вызов
    print("Testing model invocation...")
    try:
        test_response = llm.invoke("Human: Say 'Hello!'")
        print(f"Model test response: {test_response.content}")
        print("✅ Model appears to be working.")
    except Exception as e:
        print(f"❌ Error during model test invocation: {e}")
        print("   Check n_gpu_layers, model path, or Colab resources.")
        print("   If you see CUDA errors, try reducing n_gpu_layers.")
        llm = None

except Exception as e:
    print(f"\n❌ Error instantiating ChatLlamaCpp: {e}")
    print("   Ensure the model path is correct and llama-cpp-python installed correctly.")
    llm = None  # llm равен None, если создание экземпляра не удалось


✅ ChatLlamaCpp Model Instantiated Successfully.
Testing model invocation...
Model test response:  Hello! How can I assist you today?
✅ Model appears to be working.


## Импорт основных классов

In [ ]:
if llm:  # Импортировать только если загружена модель
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser, JsonOutputParser
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_core.runnables import RunnablePassthrough
    import json
    print("\n✅ LangChain core components imported.")

print("\n-- Setup Complete --")
# Успешно ли загружен llm, перед продолжением
if not llm:
    print("\n❌ LLM FAILED TO LOAD OR TEST. Cannot proceed with the workshop.")


✅ LangChain core components imported.

-- Setup Complete --


## Использование моделей и шаблонов промптов

Задача: С помощью шаблона получить ответ от LLM на основе поданного контекста

In [ ]:
# --- Models and Prompts ---
# Убедитесь, что  llm загружен
if not llm:
    print("LLM not loaded. Please run Cell 1 successfully.")
else:
    print("--- Basic Model Invocation ---")
    response = llm.invoke("Explain the concept of vector embeddings in one concise sentence.")
    print(response.content)
    print("\n")

    # --- Использование шаблонов промптов ---
    # Использование стандартного ChatPromptTemplate - ChatLlamaCpp отформатирует это
    template_string = """<s>[INST] You are a helpful assistant.
Answer the user's question based on the context provided below.
Do not use any prior knowledge. If the answer is not in the context, say you don't know.

Context: The sky is blue during the day and dark at night. Clouds can be white or grey.

Question: {user_question} [/INST]"""
    prompt_template = ChatPromptTemplate.from_template(template_string)

    # Форматирование промпта (создает список сообщений)
    formatted_prompt_messages = prompt_template.format_messages(
        user_question="What color is the sky at night?"
    )
    print("--- Formatted Prompt Messages (Input to LLM Wrapper) ---")
    print(formatted_prompt_messages)
    print("\n")

    # Вызов LLM с отформатированным промптом
    print("--- Invoking LLM with Formatted Prompt ---")
    response_with_prompt = llm.invoke(formatted_prompt_messages)
    print(response_with_prompt.content)

    print("\n--- Example 2 (Answer not in context) ---")
    response_not_in_context = llm.invoke(prompt_template.format_messages(
        user_question="What color are apples?"
    ))
    print(response_not_in_context.content)

--- Basic Model Invocation ---
 Vector embeddings are continuous representations of data points or objects in multi-dimensional space, enabling efficient processing and analysis of high-dimensional data.


--- Formatted Prompt Messages (Input to LLM Wrapper) ---
[HumanMessage(content="<s>[INST] You are a helpful assistant.\nAnswer the user's question based on the context provided below.\nDo not use any prior knowledge. If the answer is not in the context, say you don't know.\n\nContext: The sky is blue during the day and dark at night. Clouds can be white or grey.\n\nQuestion: What color is the sky at night? [/INST]", additional_kwargs={}, response_metadata={})]


--- Invoking LLM with Formatted Prompt ---
 The sky at night is typically dark. I cannot specify a particular color for the night sky as it can appear various shades of dark, depending on the presence or absence of moonlight and stars.

--- Example 2 (Answer not in context) ---
 Apples come in various colors such as red, gree

1. Модель пользуется предоставленным контекстом для ответа и пытается следовать инструкциям
2. Но модель не должна выходить за контекст и дополнять ответы
* Что означает недоработку промпта

## Использование цепочек и парсеров выходных данных

Задача: Проанализировать отзывы товаров
1. Выделить название товара
2. Выделить ключевые характеристики на основе пользовательских отзывов

In [ ]:
# --- Chains and JSON Output Parser ---
if not llm:
    print("LLM not loaded. Please run Cell 1 successfully.")
else:
    from langchain_core.output_parsers import JsonOutputParser

    # Определим парсер (мы ожидаем объект JSON)
    # JsonOutputParser для более широкой совместимости с локальными моделями
    json_parser = JsonOutputParser()

    # Создание шаблона промпта, специально запрашивая JSON
    # ПРИМЕЧАНИЕ: получение надёжного JSON из небольших локальных моделей может быть сложным.
    review_analysis_template = """<s>[INST] You are an expert product review analyzer.
Analyze the following product review and extract the product name and a list of key features mentioned.
Format your response *only* as a JSON object with keys "product_name" (string) and "key_features" (list of strings).
Do not include any other text, explanation, or markdown formatting around the JSON.

Product Review:
"{review_text}"

JSON Output: [/INST]"""

    review_prompt = ChatPromptTemplate.from_template(review_analysis_template)

    # Создание цепочки (Prompt -> LLM -> Parser)
    # Парсер попытается проанализировать текстовый вывод LLM как JSON
    review_analysis_chain = review_prompt | llm | json_parser

    # Вызов цепочки
    review1 = "The new TurboScrub 5000 is fantastic! Cleans floors spotlessly and the self-empty feature is a lifesaver. Battery lasts ages."
    review2 = "Tried the Gadgetron Mini. It's compact, which is nice, but the connection drops frequently and it feels flimsy."

    print("--- Analyzing Review 1 ---")
    try:
        result1 = review_analysis_chain.invoke({"review_text": review1})
        print(f"Type: {type(result1)}")
        print(result1)
        if isinstance(result1, dict):
            print(f"Product Name: {result1.get('product_name', 'N/A')}")
    except Exception as e:
        print(f"❌ Failed to parse output for Review 1: {e}")

    print("\n")

    print("--- Analyzing Review 2 ---")
    try:
        result2 = review_analysis_chain.invoke({"review_text": review2})
        print(f"Type: {type(result2)}")
        print(result2)
        if isinstance(result2, dict):
            print(f"Key Features: {result2.get('key_features', 'N/A')}")
    except Exception as e:
        print(f"❌ Failed to parse output for Review 2: {e}")

    # --- Альтернатива: Pydantic Parser (может быть менее надёжным с этой моделью) ---
    # Вы можете попробовать это, но это может чаще давать сбои, если JSON не идеально отформатирован

    # from pydantic import BaseModel, Field
    # from typing import List

    # class ProductInfo(BaseModel):
    #     product_name: str = Field(description="Product name")
    #     key_features: List[str] = Field(description="List of key features")

    # pydantic_parser = PydanticOutputParser(pydantic_object=ProductInfo)
    # review_prompt_pydantic = ChatPromptTemplate.from_template(
    #     review_analysis_template + "\n{format_instructions}",
    #     partial_variables={"format_instructions": pydantic_parser.get_format_instructions()}
    # )
    # pydantic_chain = review_prompt_pydantic | llm | pydantic_parser

    # try:
    #     result_pydantic = pydantic_chain.invoke({"review_text": review1})
    #     print("\n--- Pydantic Result (Review 1) ---")
    #     print(result_pydantic)
    # except Exception as e:
    #     print(f"\n❌ Pydantic parser failed for Review 1: {e}")

--- Analyzing Review 1 ---
Type: <class 'dict'>
{'product_name': 'TurboScrub 5000', 'key_features': ['Spotless floor cleaning', 'Self-empty feature', 'Long battery life']}
Product Name: TurboScrub 5000


--- Analyzing Review 2 ---
Type: <class 'dict'>
{'product_name': 'Gadgetron Mini', 'key_features': ['compact']}
Key Features: ['compact']


1. Модель действительно сгенерировала словарь
2. Модель корректно выделила названия и ключевые признаки

## Использование агентов и инструментов

In [ ]:
!pip install -U ddgs --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 65.9 MB/s eta 0:00:00


ReAct агент с веб-поиском

In [ ]:
# --- Agents and Tools (ReAct Agent with CUSTOMIZED Prompt) ---
if not llm:
    print("LLM not loaded. Please run Cell 1 successfully.")
else:
    from langchain_community.tools import DuckDuckGoSearchRun
    from langchain import hub
    from langchain.agents import create_react_agent, AgentExecutor
    from langchain_core.prompts import ChatPromptTemplate

    # Инструменты
    search_tool = DuckDuckGoSearchRun(name="duckduckgo_search")  # Explicitly name the tool
    tools = [search_tool]

    # Шаблон подсказки ReAct
    react_prompt_template = """<s>[INST] Assistant is a large language model trained by Google.

Assistant is designed to be able to assist with a wide range of tasks, from answering simple questions to
providing in-depth explanations and discussions on a wide range of topics. As a language model, Assistant is able
to generate human-like text based on the input it receives, allowing it to engage in natural-sounding
conversations and provide responses that are coherent and relevant to the topic at hand.

Assistant is constantly learning and improving, and its capabilities are constantly evolving. It is able to
process and understand large amounts of text, and can use this knowledge to provide accurate and informative
responses to a wide range of questions. Additionally, Assistant is able to generate its own text based on the
input it receives, allowing it to engage in discussions and provide explanations and descriptions on a wide range
of topics.

TOOLS:
------
Assistant has access to the following tools:
{tools}

To use a tool, please use the following format:
Thought: Do I need to use a tool? Yes
Action: The action to take. Must be exactly one of {tool_names}.
Action Input: The input to the action.
Observation: The result of the action.

When you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:
Thought: Do I need to use a tool? No
Final Answer: [your response here]

Use the 'duckduckgo_search' tool ONLY for questions that require real-time, up-to-date information (like current
events, weather, specific recent facts) or information very unlikely to be in your training data. For general
knowledge (like historical facts, definitions, chemical formulas), answer directly.

Current conversation:
{chat_history}

Question: {input}

Begin! Remember to output EITHER the Action/Action Input block OR the Final Answer, following the formats
described above EXACTLY.
Thought:[/INST] {agent_scratchpad}"""

    react_prompt = ChatPromptTemplate.from_template(react_prompt_template)

    # Частичный промпт с информацией об инструменте
    react_prompt = react_prompt.partial(
        tools=str(tools),  # Отобразить информацию об инструменте для промпта
        tool_names=", ".join([t.name for t in tools]),
    )
    print("--- Customized ReAct Prompt Ready ---")
    print(react_prompt.invoke({
        "input": "test",
        "agent_scratchpad": "",
        "chat_history": [],
        "tool_names": "duckduckgo_search",
        "tools": str(tools),
    }).to_string())

    # Агент ReAct (используя пользовательский промпт)
    agent = create_react_agent(llm, tools, react_prompt)

    # Агент-Исполнитель
    agent_executor = AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True,
        handle_parsing_errors="Please output Action or Final Answer blocks EXACTLY as shown in the instructions.",  # Дайте конкретную обратную связь по ошибкам анализа
        max_iterations=5
    )

    # Запуск Agent Executor
    print("\n--- Running Agent with CUSTOM Prompt (Current Events) ---")
    try:
        response1 = agent_executor.invoke({
            "input": "What's the latest news about Bitcoin?",
            "chat_history": []  # Убедитесь, что указано chat_history
        })
        print("\n--- Final Answer (Response 1) ---")
        print(response1.get("output", "Agent stopped without producing a final output. Check verbose logs."))
    except Exception as e:
        print(f"❌ Agent execution error: {e}")

    print("\n--- Running Agent with CUSTOM Prompt (General Knowledge) ---")
    try:
        response2 = agent_executor.invoke({
            "input": "What is the chemical formula for table salt?",
            "chat_history": []
        })
        print("\n--- Final Answer (Response 2) ---")
        print(response2.get("output", "Agent stopped without producing a final output. Check verbose logs."))
    except Exception as e:
        print(f"❌ Agent execution error: {e}")

--- Customized ReAct Prompt Ready ---
Human: <s>[INST] Assistant is a large language model trained by Google.

Assistant is designed to be able to assist with a wide range of tasks, from answering simple questions to 
providing in-depth explanations and discussions on a wide range of topics. As a language model, Assistant is able 
to generate human-like text based on the input it receives, allowing it to engage in natural-sounding 
conversations and provide responses that are coherent and relevant to the topic at hand.

Assistant is constantly learning and improving, and its capabilities are constantly evolving. It is able to 
process and understand large amounts of text, and can use this knowledge to provide accurate and informative 
responses to a wide range of questions. Additionally, Assistant is able to generate its own text based on the 
input it receives, allowing it to engage in discussions and provide explanations and descriptions on a wide range 
of topics.

TOOLS:
------
Ass

1. В случаях когда модель не обладает нужными знаниями (напр. свежими) — обращается к веб-поиску и на его основе генерирует ответ.
2. Если модель изначально обладает нужными знаниями для ответа (напр. вопрос-определение) — генерирует ответ самостоятельно.